# Driver Fatigue Detection - end-to-end

Pipeline: EDA -> precompute eye crops -> SmallEyeCNN -> CNN+LSTM -> EAR baseline -> comparison.
Датасет: FL3D (44 sessions, subject-wise eval).

## 1. Imports, config, seed

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
from collections import Counter
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from src.config import (
    SEED, EYE_SIZE, SEQ_LEN, BATCH_SIZE_CNN, BATCH_SIZE_LSTM,
    NUM_EPOCHS_CNN, NUM_EPOCHS_LSTM, LR_CNN, CLASS_NAMES,
    CHECKPOINTS_DIR, PLOTS_DIR, METRICS_DIR, EYE_CACHE_PATH,
    EAR_THRESH, PERCLOS_THRESH,
)
from src.data import (
    ensure_fl3d_dataset, index_fl3d, subject_wise_split, build_clips,
    precompute_eye_crops, load_eye_cache, get_eye_transforms, build_loaders,
)
from src.models import SmallEyeCNN, CNNLSTM
from src.training import (
    set_seed, get_device, train_loop, evaluate, compute_metrics,
    EarlyStopping, plot_history, plot_confusion, save_metrics_json,
    count_parameters, model_size_mb, measure_fps,
)
from src.inference import FaceMeshDetector, compute_ear

set_seed(SEED)
device = get_device()
print(f"Device: {device}")

Device: mps


## 2. EDA

In [2]:
root = ensure_fl3d_dataset()
frames = index_fl3d(root)
print(f"Total frames: {len(frames)}")
print(f"Subjects: {len(set(f.subject for f in frames))}")
print(f"Labels: {Counter(f.label for f in frames)}")
print(f"States: {Counter(f.state for f in frames)}")

Kaggle cache path: ~/.cache/kagglehub/datasets/matjazmuc/frame-level-driver-drowsiness-detection-fl3d/versions/1


Total frames: 53329
Subjects: 44
Labels: Counter({0: 38966, 1: 14363})
States: Counter({'alert': 38966, 'microsleep': 8869, 'yawning': 5494})


## 3. Subject-wise split

In [3]:
tr_f, va_f, te_f = subject_wise_split(frames)
print(f"train: {len(tr_f)} frames, {len(set(f.subject for f in tr_f))} subjects")
print(f"val:   {len(va_f)} frames, {len(set(f.subject for f in va_f))} subjects")
print(f"test:  {len(te_f)} frames, {len(set(f.subject for f in te_f))} subjects")

train: 44085 frames, 31 subjects
val:   5790 frames, 7 subjects
test:  3454 frames, 6 subjects


## 4. Precompute eye crops (skip if cache exists)

In [4]:
precompute_eye_crops(frames, EYE_CACHE_PATH)
cache = load_eye_cache(EYE_CACHE_PATH)
print(f"Cache: {len(cache)} / {len(frames)} ({100*len(cache)/len(frames):.1f}%)")

Cache: 52534 / 53329 (98.5%)


## 5. SmallEyeCNN training

In [5]:
train_loader, val_loader, test_loader = build_loaders(
    tr_f, va_f, te_f, cache, batch_size=BATCH_SIZE_CNN, clip=False,
)
class_counts = np.bincount([f.label for f in tr_f])
weights = torch.tensor(class_counts.sum() / (len(class_counts) * class_counts), dtype=torch.float32)
print(f"Class weights: {weights.tolist()}")
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

model = SmallEyeCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR_CNN)
history = train_loop(
    model, train_loader, val_loader, optimizer, criterion, device,
    NUM_EPOCHS_CNN,
    early_stopping=EarlyStopping(patience=5, mode="max"),
    ckpt_path=CHECKPOINTS_DIR / "small_eye_cnn.pt",
)
plot_history(history, "SmallEyeCNN", save_path=PLOTS_DIR / "smalleyecnn_history.png")

Class weights: [0.6757564544677734, 1.9224227666854858]


[01/15] train 0.3158/0.861  val 0.3511/0.889  F1 0.890  (10.1s)


[02/15] train 0.1978/0.928  val 0.3576/0.850  F1 0.854  (7.9s)


[03/15] train 0.1741/0.938  val 0.3201/0.894  F1 0.895  (7.4s)


[04/15] train 0.1545/0.945  val 0.3436/0.811  F1 0.819  (7.1s)


[05/15] train 0.1458/0.949  val 0.4008/0.758  F1 0.768  (7.9s)


[06/15] train 0.1363/0.952  val 0.4911/0.686  F1 0.697  (6.9s)


[07/15] train 0.1274/0.955  val 0.5721/0.901  F1 0.895  (6.8s)


[08/15] train 0.1215/0.956  val 0.3225/0.861  F1 0.864  (7.4s)
Early stop at epoch 8, best val F1 = 0.895


./src/training.py:226: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. SmallEyeCNN eval on test

In [6]:
model.load_state_dict(torch.load(CHECKPOINTS_DIR / "small_eye_cnn.pt"))
loss, acc, y_true_c, y_pred_c = evaluate(model, test_loader, criterion, device)
metrics_cnn = compute_metrics(y_true_c, y_pred_c, target_names=CLASS_NAMES)
print(f"SmallEyeCNN test: acc={acc:.4f}, F1={metrics_cnn['f1_weighted']:.4f}, "
      f"Recall_drowsy={metrics_cnn['recall_per_class'][1]:.4f}")
print(metrics_cnn["report"])
plot_confusion(y_true_c, y_pred_c, CLASS_NAMES, "SmallEyeCNN test",
               save_path=PLOTS_DIR / "smalleyecnn_confusion.png")
save_metrics_json(metrics_cnn, METRICS_DIR / "smalleyecnn_test.json")

SmallEyeCNN test: acc=0.8324, F1=0.8302, Recall_drowsy=0.7115
              precision    recall  f1-score   support

       alert       0.85      0.90      0.87      2241
      drowsy       0.79      0.71      0.75      1213

    accuracy                           0.83      3454
   macro avg       0.82      0.80      0.81      3454
weighted avg       0.83      0.83      0.83      3454



./src/training.py:255: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. CNN+LSTM training

In [7]:
train_loader_c, val_loader_c, test_loader_c = build_loaders(
    tr_f, va_f, te_f, cache, batch_size=BATCH_SIZE_LSTM, clip=True,
)
tr_clips = build_clips(tr_f)
clip_labels = [l for _, l in tr_clips]
clip_counts = np.bincount(clip_labels)
clip_weights = torch.tensor(clip_counts.sum() / (len(clip_counts) * clip_counts), dtype=torch.float32)
criterion_l = nn.CrossEntropyLoss(weight=clip_weights.to(device))

backbone = SmallEyeCNN()
backbone.load_state_dict(torch.load(CHECKPOINTS_DIR / "small_eye_cnn.pt"))
lstm_model = CNNLSTM(backbone).to(device)
optimizer_l = torch.optim.Adam(
    filter(lambda p: p.requires_grad, lstm_model.parameters()), lr=LR_CNN,
)
history_l = train_loop(
    lstm_model, train_loader_c, val_loader_c, optimizer_l, criterion_l, device,
    NUM_EPOCHS_LSTM,
    early_stopping=EarlyStopping(patience=5, mode="max"),
    ckpt_path=CHECKPOINTS_DIR / "cnn_lstm.pt",
)
plot_history(history_l, "CNN+LSTM", save_path=PLOTS_DIR / "cnnlstm_history.png")

[01/15] train 0.1444/0.959  val 0.2306/0.928  F1 0.928  (9.2s)


[02/15] train 0.1014/0.966  val 0.2210/0.917  F1 0.918  (8.0s)


[03/15] train 0.0913/0.970  val 0.2140/0.933  F1 0.933  (11.7s)


[04/15] train 0.0850/0.971  val 0.2040/0.930  F1 0.930  (9.7s)


[05/15] train 0.0875/0.972  val 0.2320/0.925  F1 0.926  (8.0s)


...

[08/15] train 0.0799/0.974  val 0.2147/0.913  F1 0.915  (9.2s)


[09/15] train 0.0878/0.970  val 0.2148/0.928  F1 0.928  (8.4s)


[10/15] train 0.0874/0.971  val 0.2134/0.919  F1 0.920  (8.5s)


[11/15] train 0.0759/0.976  val 0.2052/0.931  F1 0.931  (8.3s)
Early stop at epoch 11, best val F1 = 0.940


./src/training.py:226: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. CNN+LSTM eval on test

In [8]:
lstm_model.load_state_dict(torch.load(CHECKPOINTS_DIR / "cnn_lstm.pt"))
loss_l, acc_l, y_true_l, y_pred_l = evaluate(lstm_model, test_loader_c, criterion_l, device)
metrics_lstm = compute_metrics(y_true_l, y_pred_l, target_names=CLASS_NAMES)
print(f"CNN+LSTM test: acc={acc_l:.4f}, F1={metrics_lstm['f1_weighted']:.4f}, "
      f"Recall_drowsy={metrics_lstm['recall_per_class'][1]:.4f}")
print(metrics_lstm["report"])
plot_confusion(y_true_l, y_pred_l, CLASS_NAMES, "CNN+LSTM test",
               save_path=PLOTS_DIR / "cnnlstm_confusion.png")
save_metrics_json(metrics_lstm, METRICS_DIR / "cnnlstm_test.json")

CNN+LSTM test: acc=0.8663, F1=0.8610, Recall_drowsy=0.6838
              precision    recall  f1-score   support

       alert       0.85      0.96      0.90       439
      drowsy       0.91      0.68      0.78       234

    accuracy                           0.87       673
   macro avg       0.88      0.82      0.84       673
weighted avg       0.87      0.87      0.86       673



./src/training.py:255: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. EAR baseline (per-frame + PERCLOS per-window)

In [9]:
import cv2
from tqdm import tqdm

detector = FaceMeshDetector()
y_true_ear, y_pred_ear = [], []
frame_ears = {}
for f in tqdm(te_f, desc="EAR per-frame"):
    bgr = cv2.imread(str(f.path))
    if bgr is None:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    lm = detector.detect(rgb)
    if lm is None:
        continue
    le, re = compute_ear(lm)
    mean_ear = (le + re) / 2
    frame_ears[str(f.path)] = mean_ear
    y_pred_ear.append(1 if mean_ear < EAR_THRESH else 0)
    y_true_ear.append(f.label)
detector.close()
y_true_ear = np.array(y_true_ear); y_pred_ear = np.array(y_pred_ear)
metrics_ear_f = compute_metrics(y_true_ear, y_pred_ear, target_names=CLASS_NAMES)
print(f"EAR per-frame: acc={metrics_ear_f['accuracy']:.4f}, F1={metrics_ear_f['f1_weighted']:.4f}")
save_metrics_json(metrics_ear_f, METRICS_DIR / "ear_frame_test.json")
plot_confusion(y_true_ear, y_pred_ear, CLASS_NAMES, "EAR per-frame",
               save_path=PLOTS_DIR / "ear_frame_confusion.png")

# Per-window EAR+PERCLOS
test_clips = build_clips(te_f)
y_true_w, y_pred_w = [], []
for window, clip_label in test_clips:
    ears = [frame_ears.get(str(f.path)) for f in window]
    if any(e is None for e in ears):
        continue
    closed = [1 if e < EAR_THRESH else 0 for e in ears]
    perclos = sum(closed) / len(closed)
    y_pred_w.append(1 if perclos > PERCLOS_THRESH else 0)
    y_true_w.append(clip_label)
y_true_w = np.array(y_true_w); y_pred_w = np.array(y_pred_w)
metrics_ear_w = compute_metrics(y_true_w, y_pred_w, target_names=CLASS_NAMES)
print(f"EAR+PERCLOS: acc={metrics_ear_w['accuracy']:.4f}, F1={metrics_ear_w['f1_weighted']:.4f}")
save_metrics_json(metrics_ear_w, METRICS_DIR / "ear_window_test.json")
plot_confusion(y_true_w, y_pred_w, CLASS_NAMES, "EAR+PERCLOS window",
               save_path=PLOTS_DIR / "ear_window_confusion.png")

I0000 00:00:1779731916.167308 28404811 init-domain.cc:128] Fiber init: default domain = pthread, concurrency = 8, prefix = pthread-default
W0000 00:00:1779731916.171072 28404811 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.


I0000 00:00:1779731916.236251 28404811 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1779731916.243081 28404817 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779731916.259615 28404819 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


EAR per-frame:   0%|          | 0/3454 [00:00<?, ?it/s]

EAR per-frame:   0%|          | 8/3454 [00:00<00:45, 75.95it/s]

EAR per-frame:   1%|          | 18/3454 [00:00<00:39, 86.07it/s]

...

EAR per-frame: 100%|█████████▉| 3442/3454 [00:37<00:00, 95.50it/s]

EAR per-frame: 100%|█████████▉| 3452/3454 [00:37<00:00, 95.36it/s]

EAR per-frame: 100%|██████████| 3454/3454 [00:37<00:00, 92.57it/s]

EAR per-frame: acc=0.7319, F1=0.7025
EAR+PERCLOS: acc=0.7578, F1=0.7337


./src/training.py:255: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
./src/training.py:255: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Comparison table

In [10]:
rows = [
    ("EAR per-frame", "per-frame", metrics_ear_f, 0),
    ("EAR + PERCLOS", "per-window", metrics_ear_w, 0),
    ("SmallEyeCNN", "per-frame", metrics_cnn, count_parameters(SmallEyeCNN())),
    ("CNN+LSTM", "per-window", metrics_lstm, count_parameters(lstm_model)),
]
print(f"{'Method':<18} {'Level':<11} {'Acc':>5} {'F1':>5} {'Rec_D':>6} {'Prec_D':>7} {'F1_D':>5} {'Params':>8}")
for name, level, m, params in rows:
    print(f"{name:<18} {level:<11} {m['accuracy']:.3f} {m['f1_weighted']:.3f} "
          f"{m['recall_per_class'][1]:>6.3f} {m['precision_per_class'][1]:>7.3f} "
          f"{m['f1_per_class'][1]:>5.3f} {params:>8}")

Method             Level         Acc    F1  Rec_D  Prec_D  F1_D   Params
EAR per-frame      per-frame   0.732 0.702  0.364   0.740 0.488        0
EAR + PERCLOS      per-window  0.758 0.734  0.415   0.789 0.543        0
SmallEyeCNN        per-frame   0.832 0.830  0.711   0.790 0.749   101506
CNN+LSTM           per-window  0.866 0.861  0.684   0.909 0.780   280770


## 11. FPS benchmark

In [ ]:
cnn_eval = SmallEyeCNN().to(device); cnn_eval.eval()
x = torch.randn(1, 1, EYE_SIZE, EYE_SIZE, device=device)
fps_cnn = measure_fps(lambda: cnn_eval(x), n_iters=200, warmup=10)

lstm_eval = CNNLSTM(SmallEyeCNN()).to(device); lstm_eval.eval()
xs = torch.randn(1, SEQ_LEN, 1, EYE_SIZE, EYE_SIZE, device=device)
fps_lstm = measure_fps(lambda: lstm_eval(xs), n_iters=200, warmup=10)

print(f"FPS on {device}: SmallEyeCNN={fps_cnn:.0f}, CNN+LSTM={fps_lstm:.0f}")
print(f"Pipeline FPS (MediaPipe-bound): ~80")